In [7]:
import pandas as pd
import logging

from pydantic import BaseModel, Field, ValidationError
from typing import Optional


# ---------------------------------------------------------
# Validation logging
# ---------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


# ---------------------------------------------------------
# Customer 360 validation schema
# ---------------------------------------------------------

class Customer360Schema(BaseModel):
    """
    Validation schema for the customer_360 SQL table.
    """

    # Account information
    account_id: int = Field(..., gt=0)
    company_name: str = Field(..., min_length=1)
    industry: str = Field(..., min_length=1)
    country: str = Field(..., min_length=2)
    employee_count: int = Field(..., ge=0)

    # Subscription information
    signup_date: object
    plan: str = Field(..., min_length=1)
    user_count: int = Field(..., ge=0)

    admin_count: int = Field(..., ge=0)
    member_count: int = Field(..., ge=0)
    viewer_count: int = Field(..., ge=0)

    seats: int = Field(..., ge=0)

    subscription_status: str = Field(
        ...,
        min_length=1
    )

    mrr: float = Field(..., ge=0)

    started_on: object
    ended_on: Optional[object] = None

    # Usage metrics
    users_per_seat: float = Field(..., ge=0)

    subscription_duration: float = Field(
        ...,
        ge=0
    )

    # Invoice metrics
    invoice_count: int = Field(..., ge=0)

    total_invoiced: float = Field(..., ge=0)

    avg_invoice_amount: float = Field(..., ge=0)

    paid_amount: float = Field(..., ge=0)

    open_amount: float = Field(..., ge=0)

    uncollectible_amount: float = Field(..., ge=0)

    # Support metrics
    ticket_count: int = Field(..., ge=0)

    high_priority_tickets: int = Field(..., ge=0)

    urgent_tickets: int = Field(..., ge=0)

    avg_satisfaction: Optional[float] = Field(
        default=None,
        ge=0,
        le=5
    )

    avg_resolution_hours: Optional[float] = Field(
        default=None,
        ge=0
    )

    unresolved_tickets: int = Field(..., ge=0)


# ---------------------------------------------------------
# DataFrame validation function
# ---------------------------------------------------------

def validate_dataframe(
    df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:

    valid_rows = []
    corrupt_rows = []

    records = df.to_dict(orient="records")

    for idx, record in enumerate(records):

        try:

            validated_record = Customer360Schema(**record)

            valid_rows.append(
                validated_record.model_dump()
            )

        except ValidationError as error:

            record["row_index"] = idx
            record["validation_errors"] = error.errors()

            corrupt_rows.append(record)

    clean_df = pd.DataFrame(valid_rows)
    error_df = pd.DataFrame(corrupt_rows)

    logging.info(
        f"Data validation complete | "
        f"Passed: {len(clean_df)} | "
        f"Failed: {len(error_df)}"
    )

    return clean_df, error_df